<a href="https://colab.research.google.com/github/JSJeong-me/GPT-Agent/blob/main/06%20Deepagent/10_FY26_HR_Multimodal_RAG_CustomTools_DeepAgent_Colab.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# FY26 연차·평가관리 가이드 — Custom Tools → Deep Agent 실습

**1일차 PBL 심화 실습 (Google Colab)** | 대상 파일: `FY26_연차 및 평가 관련 가이드(팀장 대상).pdf`

**학습 목표**
1. PDF 멀티모달 RAG 검색을 3개의 독립적인 **LangChain Custom Tool**로 구성하고 `.invoke()`로 단독 시험
2. 같은 도구를 **LangChain Deep Agent**에 등록해 근거 검색 → 원문 확인 → 시각 검증 → 한국어 답변을 수행

원본 실습 노트북의 **페이지 텍스트 + 페이지 이미지 + Vision 설명 → Child/Parent 문서 → Chroma/MultiVectorRetriever** 방식을 계승합니다. 챗봇 응답은 PDF 근거 범위에 한정하고 출처 페이지를 표시합니다.

> **보안/비용**: 사내 인사 문서의 외부 API 전송 허용 여부를 먼저 확인하세요. 이미지 설명 생성(페이지마다 1회), 임베딩, Vision 도구 및 Agent 실행에 API 과금이 발생합니다. 수습평가의 기간·점수·인사 조치 등은 반드시 원문과 담당자가 최종 확인하세요.

## 0. 준비 — 라이브러리 설치

In [1]:
%pip -q install -U "deepagents>=0.5.4" langchain langchain-core langchain-classic langchain-openai langchain-chroma langchain-text-splitters pymupdf pillow

ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
google-colab 1.0.0 requires google-auth==2.49.0, but you have google-auth 2.62.2 which is incompatible.
gradio 6.31.0 requires pillow<13.0,>=8.0, but you have pillow 13.0.0 which is incompatible.
tweepy 4.17.0 requires oauthlib<4,>=3.2.0, but you have oauthlib 4.0.1 which is incompatible.
google-adk 2.7.1 requires google-genai<3,>=2.12.1, but you have google-genai 3.6.1 which is incompatible.
google-adk 2.7.1 requires opentelemetry-api<=1.42.1,>=1.39, but you have opentelemetry-api 1.45.2 which is incompatible.
google-adk 2.7.1 requires opentelemetry-sdk<=1.42.1,>=1.39, but you have opentelemetry-sdk 1.45.2 which is incompatible.


**설명:** Google Colab에서 위 셀을 실행하세요. 패키지 업그레이드 뒤 import 문제가 생기면 런타임을 재시작하고 API 설정 셀부터 다시 실행합니다.

## 1. API Key, 모델 및 경로

In [2]:
import os
import getpass

if not os.environ.get("OPENAI_API_KEY"):
    os.environ["OPENAI_API_KEY"] = getpass.getpass("Enter your OpenAI API Key: ")
else:
    print("OpenAI API Key is already set in the environment.")

Enter your OpenAI API Key: ··········


In [3]:
import os, getpass, json, base64, uuid
from pathlib import Path

if not os.environ.get("OPENAI_API_KEY"):
    try:
        from google.colab import userdata
        key = userdata.get("OPENAI_API_KEY")
    except Exception:
        key = None
    os.environ["OPENAI_API_KEY"] = key or getpass.getpass("OpenAI API Key: ")

ROOT = Path("/content/fy26_deepagent") if Path("/content").exists() else Path.cwd() / "fy26_deepagent"
IMAGE_DIR = ROOT / "pages"
IMAGE_DIR.mkdir(parents=True, exist_ok=True)
CHAT_MODEL = "gpt-4o-mini"   # 이미지 입력 및 tool calling 지원 모델; 계정 환경에 맞게 변경
EMBED_MODEL = "text-embedding-3-small"
MAX_PAGES = 4
print("작업 경로:", ROOT)

작업 경로: /content/fy26_deepagent


## 2. 실습 PDF 업로드

In [4]:
from pathlib import Path
if Path("/content").exists():
    from google.colab import files
    print("FY26_연차 및 평가 관련 가이드(팀장 대상).pdf 를 업로드하세요.")
    uploaded = files.upload()
    if not uploaded:
        raise RuntimeError("PDF가 업로드되지 않았습니다.")
    pdf_path = Path(next(iter(uploaded.keys()))).resolve()
else:
    pdf_path = Path("FY26_연차 및 평가 관련 가이드(팀장 대상).pdf").resolve()

if not pdf_path.is_file() or pdf_path.suffix.lower() != ".pdf":
    raise FileNotFoundError(f"PDF 파일을 확인해 주세요: {pdf_path}")
print("실습 PDF:", pdf_path.name)

FY26_연차 및 평가 관련 가이드(팀장 대상).pdf 를 업로드하세요.


Saving FY26_연차 및 평가 관련 가이드(팀장 대상).pdf to FY26_연차 및 평가 관련 가이드(팀장 대상).pdf
실습 PDF: FY26_연차 및 평가 관련 가이드(팀장 대상).pdf


## 3. PDF를 페이지별 텍스트 + PNG로 분해

In [5]:
import fitz
pages = []
with fitz.open(str(pdf_path)) as doc:
    for i, p in enumerate(doc):
        pg = i + 1
        img_path = IMAGE_DIR / f"page_{pg:02d}.png"
        p.get_pixmap(matrix=fitz.Matrix(1.5, 1.5), alpha=False).save(str(img_path))
        pages.append({"page": pg, "text": p.get_text("text", sort=True).strip(), "image": str(img_path)})
print("페이지 수:", len(pages))
for p in pages:
    print(f"p.{p['page']} 텍스트 길이={len(p['text'])}")
assert pages, "빈 PDF 입니다."

/usr/local/lib/python3.13/dist-packages/fitz/__init__.py:17: FutureWarning: The `fitz` API is deprecated and will be removed in future. Use `import pymupdf` instead.
  warnings.warn(


페이지 수: 9
p.1 텍스트 길이=24
p.2 텍스트 길이=314
p.3 텍스트 길이=111
p.4 텍스트 길이=111
p.5 텍스트 길이=106
p.6 텍스트 길이=106
p.7 텍스트 길이=106
p.8 텍스트 길이=150
p.9 텍스트 길이=6


## 4. 슬라이드 Vision 설명 생성 — 이미지 안의 표/도식 검색 준비

In [6]:
from langchain_openai import ChatOpenAI
from langchain_core.messages import HumanMessage, SystemMessage

vision_llm = ChatOpenAI(model=CHAT_MODEL, temperature=0)

def image_data_uri(image_path: str) -> str:
    with open(image_path, "rb") as f:
        return "data:image/png;base64," + base64.b64encode(f.read()).decode("ascii")

def describe_page(p: dict) -> str:
    prompt = (
        f"이 이미지는 FY26 사내 연차·평가관리 가이드의 {p['page']}페이지입니다. "
        "보이는 문장, 표, 화살표, 평가 절차와 숫자/기한을 한국어로 충실히 기록하세요. "
        "읽을 수 없는 내용은 추정하지 말고 [판독 불가]라고 표시하세요. "
        "원문에 없는 규정은 추가하지 마세요."
    )
    response = vision_llm.invoke([HumanMessage(content=[
        {"type": "text", "text": prompt},
        {"type": "image_url", "image_url": {"url": image_data_uri(p['image'])}}
    ])])
    return str(response.content)

In [7]:
# 1회 실행 시 페이지 수만큼 유료 Vision 호출 발생
slide_summaries = {p["page"]: describe_page(p) for p in pages}
print("Vision 설명 생성 완료:", len(slide_summaries))
print(slide_summaries[1][:500])

Vision 설명 생성 완료: 9
- 제목: “FY-26 연차사용 및 평가관리 가이드”
- 하단: 중앙정보기술

[판독 불가]


## 5. MultiVectorRetriever: 검색용 Child / 근거용 Parent

텍스트 청크와 Vision 요약 각각을 **child**로 임베딩하고, 반환 결과는 페이지 단위 **parent**로 유지합니다. 이미지 경로 자체를 임베딩하지 않습니다.

In [8]:
from langchain_core.documents import Document
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_openai import OpenAIEmbeddings
from langchain_chroma import Chroma
from langchain_core.stores import InMemoryStore
from langchain_classic.retrievers.multi_vector import MultiVectorRetriever

splitter = RecursiveCharacterTextSplitter(chunk_size=850, chunk_overlap=100)
children, parents = [], []
for p in pages:
    pid = str(uuid.uuid4())
    parent = Document(
        page_content=(p['text'] + "\n\n[Vision 이미지 설명]\n" + slide_summaries[p['page']]).strip(),
        metadata={"doc_id":pid, "page":p['page'], "source":pdf_path.name, "image_path":p['image']}
    )
    parents.append((pid, parent))
    for source_type, content in [("pdf_text", p['text']), ("vision_description", slide_summaries[p['page']])]:
        if not content.strip():
            continue
        for chunk in splitter.split_text(content):
            children.append(Document(page_content=chunk, metadata={"doc_id":pid, "page":p['page'], "source_type":source_type}))

# Colab 실행마다 고유 Collection: 중복 색인 방지
collection_name = "fy26_deepagent_" + uuid.uuid4().hex[:10]
vectorstore = Chroma(collection_name=collection_name, embedding_function=OpenAIEmbeddings(model=EMBED_MODEL))
docstore = InMemoryStore()
docstore.mset(parents)
vectorstore.add_documents(children)
retriever = MultiVectorRetriever(vectorstore=vectorstore, docstore=docstore, id_key="doc_id", search_kwargs={"k":8})
page_index = {p['page']: p for p in pages}
print(f"Parent: {len(parents)}, Child: {len(children)}")

Parent: 9, Child: 18


# Part 1. LangChain Custom Tools 생성 및 단독 테스트

도구의 책임을 분리합니다.
- `search_fy26_pages`: 질문과 관련된 페이지 번호와 텍스트 단서 검색
- `read_fy26_page`: 지정 페이지의 PDF 텍스트와 Vision 설명을 읽음
- `inspect_fy26_page_visual`: 원본 페이지 이미지에서 특정 질문의 근거를 Vision 모델로 직접 확인

**중요:** 도구 결과는 문서의 증거이고, 법적·인사적 최종 판단은 아닙니다.

In [9]:
from langchain_core.tools import tool

@tool
def search_fy26_pages(query: str) -> str:
    """FY26 연차·평가관리 PDF에서 질문과 관련된 근거 페이지를 검색한다. 답변하기 전에 먼저 호출한다."""
    docs = retriever.invoke(query)
    unique = []
    seen = set()
    for d in docs:
        pg = int(d.metadata['page'])
        if pg not in seen:
            unique.append({"page": pg, "snippet":d.page_content[:650]})
            seen.add(pg)
        if len(unique) >= MAX_PAGES:
            break
    return json.dumps({"query":query, "hits":unique}, ensure_ascii=False)

@tool
def read_fy26_page(page_number: int) -> str:
    """FY26 가이드의 특정 1개 페이지 텍스트 및 이미지 설명을 반환한다. page_number는 1부터 시작한다."""
    if page_number not in page_index:
        return json.dumps({"error":f"페이지 범위는 1~{len(pages)}입니다."}, ensure_ascii=False)
    p = page_index[page_number]
    return json.dumps({
        "page":page_number,
        "pdf_text":p['text'][:4500],
        "vision_description":slide_summaries[page_number][:4500],
        "source":pdf_path.name
    }, ensure_ascii=False)

@tool
def inspect_fy26_page_visual(page_number: int, question: str) -> str:
    """FY26 가이드 원본 슬라이드 이미지 1페이지에서 question에 대한 사실·표·수치·분기를 시각적으로 검증한다."""
    if page_number not in page_index:
        return json.dumps({"error":f"페이지 범위는 1~{len(pages)}입니다."}, ensure_ascii=False)
    p = page_index[page_number]
    msg = HumanMessage(content=[
        {"type":"text", "text":(
            f"FY26 사내 가이드의 p.{page_number} 원본 이미지에만 근거하여 답하세요. "
            f"질문: {question}\n숫자·기간·조건·분기가 보이면 정확히 기록하고, "
            "보이지 않거나 판독 불가이면 확인 불가라고 명시하세요. 외부 규정을 추가하지 마세요.")},
        {"type":"image_url", "image_url":{"url":image_data_uri(p['image'])}}
    ])
    reply = vision_llm.invoke([msg])
    return json.dumps({"page":page_number,"question":question,"visual_evidence":str(reply.content)}, ensure_ascii=False)


In [10]:
fy26_tools = [search_fy26_pages, read_fy26_page, inspect_fy26_page_visual]
print([(t.name, t.description[:65]) for t in fy26_tools])

[('search_fy26_pages', 'FY26 연차·평가관리 PDF에서 질문과 관련된 근거 페이지를 검색한다. 답변하기 전에 먼저 호출한다.'), ('read_fy26_page', 'FY26 가이드의 특정 1개 페이지 텍스트 및 이미지 설명을 반환한다. page_number는 1부터 시작한다.'), ('inspect_fy26_page_visual', 'FY26 가이드 원본 슬라이드 이미지 1페이지에서 question에 대한 사실·표·수치·분기를 시각적으로 검증한다.')]


## 6. Custom Tool 단독 테스트 ①: 의미 검색 (Retriever)

In [11]:
q = "수습평가 진행 절차와 평가 기준은 무엇인가요?"
search_result = json.loads(search_fy26_pages.invoke({"query": q}))
print(json.dumps(search_result, ensure_ascii=False, indent=2))
assert "hits" in search_result
assert all(1 <= x["page"] <= len(pages) for x in search_result["hits"])
print("검색 도구 형식 검사: PASS")

{
  "query": "수습평가 진행 절차와 평가 기준은 무엇인가요?",
  "hits": [
    {
      "page": 7,
      "snippet": "2. 평가관리\n\n\n\n\n\n                                                                                        - 7 -\n\n[Vision 이미지 설명]\n```\n2. 평가관리\n\n수습 평가 공동원칙\n\n직무·조직 적합성 판단\n- 해당 직무 수행 가능성 확인\n- 조직 문화 적응도 평가\n- 장기적 성장 가능성 검토\n- 팀 내 역할 적합성 판단\n\n주요 평가 포인트\n- 업무 이해도와 속도\n- 지식 이행 및 업무 완성도\n- 근태 및 기본 태도\n- 팀원과의 협업력\n\n평가 시 유의사항\n- 첫인상에 의한 판단 지양\n- 단순 실수 과대평가 금지\n- 수습기간 내 집중 관찰\n  (정해진 기간 내 평가 필수 - 규정 수습기간 3개월)\n- 객관적 기준 적용\n```"
    },
    {
      "page": 5,
      "snippet": "2. 평가관리\n\n\n\n\n\n                                                                                        - 5 -\n\n[Vision 이미지 설명]\n```\n2. 평가관리\n\n평가자로서의 팀장 역할\n팀원 평가는 팀장의 핵심 관리 역할이자 책임입니다.\n\n평가 기준 통일의 중요성\n- 모든 팀원에게 일관된 기준 적용\n- 객관적 평가 지표 확립\n- 평가 기준에 대한 사전 공유\n- 팀 내 평가 일관성 유지\n\n공정한 평가와 피드백\n- 사실에 기반한 객관적 평가\n- 구체적 사례 중심 피드백\n- 성장 지향적 피드백 제공\n- 정기평가 일정 준수\n  (임직원평가: 년 1회 / 역량평가: 분기 1회 / 수습평가: 주 1회(총 12회))\n  ※ 기한 내 평

## 7. Custom Tool 단독 테스트 ②: 근거 페이지 읽기

In [ ]:
# 검색 결과의 첫 페이지를 사용하므로 고정 페이지를 가정하지 않습니다.
if not search_result['hits']:
    raise RuntimeError("검색 결과가 없습니다. 임베딩/색인 생성 상태를 확인하세요.")
selected_page = search_result['hits'][0]['page']
read_result = json.loads(read_fy26_page.invoke({"page_number":selected_page}))
print("p.", selected_page, "\n", read_result['pdf_text'][:700])
print("[Vision 설명]\n", read_result['vision_description'][:900])
assert read_result['page'] == selected_page
print("페이지 읽기 도구 형식 검사: PASS")

In [ ]:
pages_retrieved = [hit['page'] for hit in search_result['hits']]
selected_page = search_result['hits'][0]['page']
read_result = json.loads(read_fy26_page.invoke({"page_number":selected_page}))
print("p.", selected_page, "\n", read_result['pdf_text'][:700])
print("[Vision 설명]\n", read_result['vision_description'][:900])
assert read_result['page'] == selected_page
print("페이지 읽기 도구 형식 검사: PASS")
print("검색된 모든 페이지 목록:", pages_retrieved)

## 8. Custom Tool 단독 테스트 ③: 원본 이미지 확인 (API 호출)

In [23]:
visual = json.loads(inspect_fy26_page_visual.invoke({
    "page_number": selected_page,
    "question": "이 페이지에 표시된 평가/결재 관련 항목과 숫자, 주의사항은 무엇인가요?"
}))
print(json.dumps(visual, ensure_ascii=False, indent=2))
assert visual['page'] == selected_page
print("원본 이미지 확인 도구 형식 검사: PASS (내용 정확성은 수동 검증)")

{
  "page": 7,
  "question": "이 페이지에 표시된 평가/결재 관련 항목과 숫자, 주의사항은 무엇인가요?",
  "visual_evidence": "확인 불가."
}
원본 이미지 확인 도구 형식 검사: PASS (내용 정확성은 수동 검증)


## 9. Custom Tool 예외 처리 테스트

실습 중 잘못된 페이지를 요청해도 전체 코드가 중단되지 않도록 오류 메시지를 반환합니다.

In [24]:
bad = json.loads(read_fy26_page.invoke({"page_number":9999}))
print(bad)
assert 'error' in bad
print("예외 처리: PASS")

{'error': '페이지 범위는 1~9입니다.'}
예외 처리: PASS


# Part 2. LangChain Deep Agent + Custom Tools 통합

Deep Agent가 검색 도구를 먼저 호출하고, 필요한 페이지를 읽은 다음, **도형·표·수치처럼 시각적 증거가 필요한 경우** Vision 확인 도구를 호출하도록 지시합니다. 검색 결과가 곧 정답은 아니며, 문서 밖 내용은 판단을 보류합니다.

In [25]:
from deepagents import create_deep_agent

SYSTEM_PROMPT = """당신은 FY26 연차사용 및 평가관리 문서의 근거 기반 실습용 도우미입니다.
1) FY26 문서 관련 질문을 받으면 반드시 search_fy26_pages를 먼저 실행하세요.
2) 검색된 페이지 중 관련 페이지에 read_fy26_page를 실행해 실제 근거를 확인하세요.
3) 표, 도형, 평가 프로세스, 점수, 기간, 결재권한의 정확한 확인이 필요하면 inspect_fy26_page_visual을 사용하세요.
4) 도구로 확인된 정보만 답하고 각 사실에 (p.N) 형식의 출처를 표시하세요.
5) 근거가 없거나 모호하면 '이 문서만으로 확인할 수 없습니다'라고 답하세요.
6) 문서 외부의 노동법, 사내정책, 평가기준을 만들어 내지 마세요.
7) 원문을 그대로 장황하게 복제하지 말고 간결한 한국어로 요약하세요.
8) 개인정보 및 민감한 인사정보를 추론하지 마세요.
"""

# 모델 문자열 대신 이미 생성한 LangChain ChatOpenAI 인스턴스를 활용
agent_model = ChatOpenAI(model=CHAT_MODEL, temperature=0)
fy26_agent = create_deep_agent(
    model=agent_model,
    tools=fy26_tools,
    system_prompt=SYSTEM_PROMPT,
)
print("Deep Agent 구성 완료. 등록 도구:", [t.name for t in fy26_tools])

Deep Agent 구성 완료. 등록 도구: ['search_fy26_pages', 'read_fy26_page', 'inspect_fy26_page_visual']


In [ ]:
fy26_agent.invoke({"messages":[{"role":"user", "content":"연차 승인에 최종 책임자는?."}]})

In [ ]:
fy26_agent.invoke({"messages":[{"role":"user", "content":"수습평가 절차를 설명해 주세요. 중요한 숫자와 분기는 원본 이미지를 확인하세요."}]})

In [29]:
fy26_agent.invoke({"messages":[{"role":"user", "content":"회사규정에 여름휴가 관한 내용을 알려 주세요."}]})

{'messages': [HumanMessage(content='회사규정에 여름휴가 관한 내용을 알려 주세요.', additional_kwargs={}, response_metadata={}, id='20c17066-340b-4062-8ae5-2e7b7364cab5'),
  AIMessage(content='이 문서만으로 확인할 수 없습니다.', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 11, 'prompt_tokens': 2377, 'total_tokens': 2388, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 0, 'rejected_prediction_tokens': 0}, 'prompt_tokens_details': {'audio_tokens': 0, 'cache_write_tokens': None, 'cached_tokens': 1920}}, 'model_provider': 'openai', 'model_name': 'gpt-4o-mini-2024-07-18', 'system_fingerprint': 'fp_e28d05b531', 'id': 'chatcmpl-EloihRuNHTZwIrIphr5aQxOLEsBpH', 'service_tier': 'default', 'finish_reason': 'stop', 'logprobs': None}, id='lc_run--01a1f237-93cd-76ff-95f0-2da0868bcf66-0', tool_calls=[], invalid_tool_calls=[], usage_metadata={'input_tokens': 2377, 'output_tokens': 11, 'total_tokens': 2388, 'input_token_details': {'aud

## 10. Deep Agent 질의응답: 검색 → 도구 호출 → 답변

In [30]:
def ask_agent(question: str, show_trace: bool = True):
    result = fy26_agent.invoke(
        {"messages":[{"role":"user", "content":question}]},
        config={"recursion_limit":35}
    )
    msgs = result["messages"]
    if show_trace:
        for m in msgs:
            calls = getattr(m, "tool_calls", None)
            if calls:
                for call in calls:
                    print("[Tool Call]", call.get("name"), call.get("args"))
            if getattr(m, "type", "") == "tool":
                print("[Tool Result]", getattr(m,"name",""), str(m.content)[:280])
    final = msgs[-1].content
    print("\n[Agent 최종 답변]\n", final)
    return result



In [31]:
agent_result = ask_agent("수습평가 절차를 문서 근거와 페이지 번호로 설명해 주세요. 중요한 숫자와 분기는 원본 이미지를 확인하세요.")

[Tool Call] search_fy26_pages {'query': '수습평가 절차'}
[Tool Result] search_fy26_pages {"query": "수습평가 절차", "hits": [{"page": 7, "snippet": "2. 평가관리\n\n\n\n\n\n                                                                                        - 7 -\n\n[Vision 이미지 설명]\n```\n2. 평가관리\n\n수습 평가 공동원칙\n\n직무·조직 적합성 판단\n- 해당 직무 수행 가능성 확인\n- 조직 문화 적응도 평가\n- 장기적 성장 가능성 검
[Tool Call] read_fy26_page {'page_number': 7}
[Tool Call] read_fy26_page {'page_number': 8}
[Tool Call] inspect_fy26_page_visual {'page_number': 8, 'question': '수습평가 절차'}
[Tool Result] read_fy26_page {"page": 7, "pdf_text": "2. 평가관리\n\n\n\n\n\n                                                                                        - 7 -", "vision_description": "```\n2. 평가관리\n\n수습 평가 공동원칙\n\n직무·조직 적합성 판단\n- 해당 직무 수행 가능성 확인\n- 조직 문화 적응도 평가\n- 장기적 성장 가능성 검토\n- 팀 내 역할 적합성 판단\n\n주요
[Tool Result] read_fy26_page {"page": 8, "pdf_text": "2. 평가관리\n\n\n\n\n\n   - 수습평가: erp 인사관리– 평가관리화면/ 기한내반드시평가완료할것\n\n\n                                   

In [32]:
ask_agent("연차 승인에 최종 책임자는?")

[Tool Call] search_fy26_pages {'query': '연차 승인 최종 책임자'}
[Tool Result] search_fy26_pages {"query": "연차 승인 최종 책임자", "hits": [{"page": 2, "snippet": "목  차\n\n\n\n\n      1. 연차결재및주의사항                               2. 평가관리\n\n\n1.  결재전체크                            1  평가자로서의팀장역할\n\n\n2.  연차결재원칙                           2   팀원(업적/ 역량)평가\n\n\n                              
[Tool Call] read_fy26_page {'page_number': 4}
[Tool Result] read_fy26_page {"page": 4, "pdf_text": "1. 연차결재및주의사항\n\n\n\n\n\n                                                                                        - 4 -", "vision_description": "1. 연차결재 및 주의사항\n\n**연차 결재 원칙**\n\n**연차발생**\n- 연차는 법정근로기준법에 따라 1년 미만은 월 1일 / 1년 이상은 15일로 보장 (2년마다 1개월 가산 최대 25일)\

[Agent 최종 답변]
 연차 승인에 대한 최종 책임자는 각 본부장입니다. 팀장은 재량에 따라 결정할 수 없으며, 결재된 결재라인을 반드시 확인해야 합니다(p.4).


{'messages': [HumanMessage(content='연차 승인에 최종 책임자는?', additional_kwargs={}, response_metadata={}, id='87897c9a-968b-48bc-ba3a-4cf6f4fe271b'),
  AIMessage(content='', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 23, 'prompt_tokens': 2373, 'total_tokens': 2396, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 0, 'rejected_prediction_tokens': 0}, 'prompt_tokens_details': {'audio_tokens': 0, 'cache_write_tokens': None, 'cached_tokens': 0}}, 'model_provider': 'openai', 'model_name': 'gpt-4o-mini-2024-07-18', 'system_fingerprint': 'fp_e0dcbbf9dc', 'id': 'chatcmpl-EloqNwAgec43nWpIoVPPDocf9cxtT', 'service_tier': 'default', 'finish_reason': 'tool_calls', 'logprobs': None}, id='lc_run--01a1f23e-d810-72d1-9bc6-46b6ca6e05d6-0', tool_calls=[{'name': 'search_fy26_pages', 'args': {'query': '연차 승인 최종 책임자'}, 'id': 'call_BdqLm8FTYa4CwvpFFQHXylOb', 'type': 'tool_call'}], invalid_tool_calls=[], usage_metad

In [33]:
ask_agent("회사규정에 여름휴가 관한 내용을 알려 주세요.")

[Tool Call] search_fy26_pages {'query': '여름휴가'}
[Tool Result] search_fy26_pages {"query": "여름휴가", "hits": [{"page": 9, "snippet": "감사합니다.\n\n[Vision 이미지 설명]\n[판독 불가]"}, {"page": 3, "snippet": "1. 연차결재및주의사항\n\n\n\n\n\n                                                                                        - 3 -\n\n[Vision 이미지 설명]\n1. 연차결재 및 주의사항\n\n연차 결재 전 체크사
[Tool Call] read_fy26_page {'page_number': 3}
[Tool Call] read_fy26_page {'page_number': 4}
[Tool Result] read_fy26_page {"page": 3, "pdf_text": "1. 연차결재및주의사항\n\n\n\n\n\n                                                                                        - 3 -", "vision_description": "1. 연차결재 및 주의사항\n\n연차 결재 전 체크사항\n\n팀/프로젝트 일정 확인\n- 팀 내 주요일정 검토\n- 팀 내 중복 휴가 여부\n- 고객/협업 부서 일정\n- 계절적 업무 피크 고려\n\n
[Tool Result] read_fy26_page {"page": 4, "pdf_text": "1. 연차결재및주의사항\n\n\n\n\n\n                                                                                        - 4 -", "vision_description": "1. 연차결재 및 주의사항\n\n**연차 결재 원칙**\n\n**연차발생

{'messages': [HumanMessage(content='회사규정에 여름휴가 관한 내용을 알려 주세요.', additional_kwargs={}, response_metadata={}, id='28446c21-57f2-4af6-ac02-8dba9c40b4bb'),
  AIMessage(content='', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 20, 'prompt_tokens': 2377, 'total_tokens': 2397, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 0, 'rejected_prediction_tokens': 0}, 'prompt_tokens_details': {'audio_tokens': 0, 'cache_write_tokens': None, 'cached_tokens': 2304}}, 'model_provider': 'openai', 'model_name': 'gpt-4o-mini-2024-07-18', 'system_fingerprint': 'fp_e0dcbbf9dc', 'id': 'chatcmpl-ElornqQEGnexTc5AH8fSie3hQ0XDd', 'service_tier': 'default', 'finish_reason': 'tool_calls', 'logprobs': None}, id='lc_run--01a1f240-2d9b-73c9-bab4-68ab96db3dc6-0', tool_calls=[{'name': 'search_fy26_pages', 'args': {'query': '여름휴가'}, 'id': 'call_ta5dItzoCcGJ4aEHzbNYP87b', 'type': 'tool_call'}], invalid_tool_calls=[], usage_

## 11. Deep Agent 판단 보류 테스트

In [34]:
abstention_result = ask_agent("이 회사의 2027년 연차수당 지급액은 얼마인가요? 문서에 없다면 확인 불가라고 답하세요.", show_trace=False)
# LLM 의미 판단은 assert로 강제하지 않고, 원본에 근거가 없는 정보를 생성했는지 사람이 검토합니다.


[Agent 최종 답변]
 이 문서에서는 2027년 연차수당 지급액에 대한 정보가 확인되지 않습니다.


## 12. 실습용 평가 질문과 체크리스트

In [39]:
evaluation_cases = [
    {"question":"연차 결재 전에 확인해야 할 사항은 무엇인가요?", "expected_pages":[3,4]},
    # {"question":"업적·역량 평가 시 팀장이 확인할 사항은 무엇인가요?", "expected_pages":[5,6]},
    # {"question":"수습평가 단계별 절차를 설명하세요.", "expected_pages":[7,8]},
    # {"question":"2027년 연차수당 금액이 문서에 있나요?", "expected_pages":[]},
]
for item in evaluation_cases:
    result = json.loads(search_fy26_pages.invoke({"query":item['question']}))
    got = [x['page'] for x in result['hits']]
    expected = item['expected_pages']
    print({"question":item['question'], "retrieved_pages":got,
           "expected_pages_hint":expected,
           "expected_hit": bool(set(got)&set(expected)) if expected else "해당 없음(근거 부재 검증 필요)"})
print("주의: 예상 페이지는 기존 교육자료 기준의 점검 힌트이며, 정답/정확도 자동 판정이 아닙니다.")

{'question': '연차 결재 전에 확인해야 할 사항은 무엇인가요?', 'retrieved_pages': [3, 4, 2, 1], 'expected_pages_hint': [3, 4], 'expected_hit': True}
주의: 예상 페이지는 기존 교육자료 기준의 점검 힌트이며, 정답/정확도 자동 판정이 아닙니다.


## 13. 학습 정리와 확장 과제

| 구분 | Part 1: Custom Tool | Part 2: Deep Agent |
|---|---|---|
| 실행 주체 | 학습자가 도구 직접 `.invoke()` | Agent가 질문에 따라 도구 선택 |
| 검색 | `search_fy26_pages` | 동일 Tool 재사용 |
| 근거 확인 | `read_fy26_page` | 동일 Tool 재사용 |
| 도표·수치 확인 | `inspect_fy26_page_visual` | 필요한 경우 호출하도록 지시 |
| 결과 | JSON 도구 응답 | 근거 페이지 표시된 한국어 답변 |

**확장 과제:** `Structured Output`으로 `checklist_item, requirement, source_page, verification_status`를 정의하고, 검증이 끝난 항목만 Excel로 출력하는 `export_checklist` 도구를 추가해 보세요. 원본 PDF에 없는 사내 규칙이나 실제 평가 점수는 생성하지 마세요.

**실습 검증 기준:** 도구가 입력 스키마를 준수하는가? 검색된 페이지와 원본이 일치하는가? 도형 속 기간/점수 분기를 잘못 해석하지 않는가? 문서 외 질문에 판단을 보류하는가?

**참고:** [LangChain Deep Agents](https://docs.langchain.com/oss/python/deepagents/overview), [LangChain Tools](https://docs.langchain.com/oss/python/langchain/tools). Colab 런타임·패키지 버전에 따라 라이브러리 API가 달라질 수 있습니다. 외부 API 호출을 포함하는 전체 셀은 이 환경에서 실제 Colab 실행까지 검증하지 않았습니다.